In [ ]:
-- 03_GOLD_INSIGHTS.SQL
-- Layer: Gold (BI-Ready)
-- Purpose: Aggregations, joins, and business logic.
-- These tables power the Power BI dashboard directly.

-- Step 1: Create Gold Schema
CREATE SCHEMA IF NOT EXISTS workspace.gold;

-- Table 1: Churn Breakdown
CREATE OR REPLACE TABLE workspace.gold.churn_breakdown AS
SELECT
    SUM(CASE WHEN churned = 0 THEN 1 ELSE 0 END) AS active_members,
    SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) AS churned_members
FROM workspace.silver.members;

-- Table 2: Churn by Persona
CREATE OR REPLACE TABLE workspace.gold.churn_by_persona AS
SELECT
    persona,
    COUNT(*) AS total_members,
    SUM(CASE WHEN churned = 0 THEN 1 ELSE 0 END) AS active_count,
    SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) AS churned_count,
    ROUND(
        SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS churn_percent
FROM workspace.silver.members
GROUP BY persona
ORDER BY churn_percent DESC;

-- Table 3: Average Balance by Status
CREATE OR REPLACE TABLE workspace.gold.avg_balance_per_status AS
SELECT
    CASE
        WHEN m.churned = 1 THEN 'Churned'
        ELSE 'Active'
    END AS churn_status,
    ROUND(AVG(a.balance), 2) AS avg_balance,
    COUNT(DISTINCT m.member_id) AS total_members
FROM workspace.silver.members m
JOIN workspace.silver.accounts a
    ON m.member_id = a.member_id
GROUP BY churn_status;

-- Table 4: Transaction Summary by Persona
CREATE OR REPLACE TABLE workspace.gold.transaction_summary AS
WITH member_txn_summary AS (
    SELECT
        m.member_id,
        m.persona,
        m.churned,
        COUNT(t.transaction_id) AS total_transactions,
        SUM(t.amount) AS total_spent,
        AVG(t.amount) AS avg_transaction_amount
    FROM workspace.silver.members m
    LEFT JOIN workspace.silver.transactions t
        ON m.member_id = t.member_id
    GROUP BY m.member_id, m.persona, m.churned
)
SELECT
    persona,
    COUNT(*) AS total_members,
    SUM(total_transactions) AS total_transactions,
    ROUND(AVG(total_spent), 2) AS avg_total_spent,
    ROUND(AVG(avg_transaction_amount), 2) AS avg_transaction_amount,
    ROUND(
        SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS churn_percent
FROM member_txn_summary
GROUP BY persona
ORDER BY churn_percent DESC;